# Time-series disturbance detection

Runnable walk-through of the `disturb` pipeline. The pure-numpy
core (decompose + detect) needs only numpy; the cube build requires the
full geospatial stack (`pixi install`).

Sequence: simulate a pixel series, decompose into trend/seasonal/residual,
detect the breakpoint, then sketch the validation step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from disturb.decompose import harmonic_decompose
from disturb.detect import detect_breakpoint

## 1. A synthetic disturbed pixel

Trend + annual cycle + noise, with a fire-like NDVI drop injected partway
through. (For real data, this series comes from `disturb.cube.build_ndvi_cube`.)

In [ ]:
rng = np.random.default_rng(42)
period = 365.25
t = np.arange(0, 5 * 365, 16, dtype=float)
times = np.datetime64('2018-01-01') + t.astype('timedelta64[D]')
break_at = int(t.size * 0.62)

seasonal = 0.25 * np.sin(2 * np.pi * t / period)
ndvi = 0.6 + seasonal + rng.normal(0, 0.02, t.size)
ndvi[break_at + 1:] -= 0.35  # disturbance

plt.figure(figsize=(9, 3))
plt.plot(times, ndvi, '.', ms=4)
plt.title('Synthetic NDVI series with injected disturbance')
plt.ylabel('NDVI'); plt.tight_layout()

## 2. Seasonal-trend decomposition

In [ ]:
fit = harmonic_decompose(ndvi, t=t, period=period, n_harmonics=2)
print('seasonal amplitude:', round(fit.seasonal_amplitude(1), 3))

fig, ax = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
ax[0].plot(times, fit.trend); ax[0].set_ylabel('trend')
ax[1].plot(times, fit.seasonal); ax[1].set_ylabel('seasonal')
ax[2].plot(times, fit.residual); ax[2].set_ylabel('residual')
plt.tight_layout()

## 3. Breakpoint detection on the residual

In [ ]:
bp = detect_breakpoint(fit.residual, times=times, min_segment=5)
print('detected:', bp.detected)
print('break date:', bp.date)
print('magnitude:', round(bp.magnitude, 3))
print('score:', round(bp.score, 3))

plt.figure(figsize=(9, 3))
plt.plot(times, ndvi, '.', ms=4)
plt.axvline(bp.date, color='r', ls='--', label='detected break')
plt.legend(); plt.title('Detected disturbance'); plt.tight_layout()

## 4. Validation (sketch)

Stack the per-pixel `bp.date` / `bp.magnitude` into 2-D maps, then call
`disturb.validate.spatial_agreement(dates, magnitude, event_mask, event_date)`
to get the detection rate and false-alarm rate against a documented event
(see `config/aoi.yaml`).